# 01. Computer Vision: Sudoku Grid Detection & Perspective Rectification

### Overview
This notebook implements the complete Computer Vision (CV) pipeline required to localize, extract, and rectify a 9×9 Sudoku grid from real-world photos or screenshots.

### Key Algorithmic Stages:
1. **Preprocessing**: Grayscale conversion and Gaussian Adaptive Thresholding to handle uneven lighting and shadows.
2. **Contour Extraction**: Detecting the largest convex quadrilateral polygon representing the puzzle border.
3. **Corner Ordering**: Sorting the 4 vertices into standard clockwise order (`[top-left, top-right, bottom-right, bottom-left]`).
4. **Homography & Perspective Transform**: Unwarping the detected bounding quad into a normalized $900 \times 900$ square.
5. **Cell Extraction**: Segmenting the rectified grid into 81 uniform cell patches with margin filtering to strip border noise.

In [ ]:
import os
import sys
from pathlib import Path

# Dynamically locate project root
def find_root() -> Path:
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / "models" / "digit_classifier.keras").exists():
            return parent
    return Path(os.path.abspath("..")).resolve()

PROJECT_ROOT = find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt

# Matplotlib high-contrast styling
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

print("OpenCV Version:", cv2.__version__)
print("Project Root:", PROJECT_ROOT)

## 1. Image Ingestion
Load a sample Sudoku image from `data/raw/` or `data/test_set/` and inspect its dimensions.

In [ ]:
# Load raw test image
sample_path = PROJECT_ROOT / "data" / "raw" / "sample3.png"
if not sample_path.exists():
    sample_path = PROJECT_ROOT / "data" / "test_set" / "sample1.png"

img = cv2.imread(str(sample_path))
if img is None:
    raise FileNotFoundError(f"Image not found at {sample_path}")

print(f"Loaded image: {sample_path.name} | Shape: {img.shape}")

plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title(f"Raw Input: {sample_path.name}")
plt.axis("off")
plt.show()

## 2. Preprocessing: Grayscale & Adaptive Thresholding
Sudoku photos often suffer from shadows, camera flare, or paper texture. 
We apply **Gaussian Blur** followed by **Adaptive Gaussian Thresholding** (`THRESH_BINARY_INV`) to isolate grid borders cleanly.

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (9, 9), 0)

thresh = cv2.adaptiveThreshold(
    blurred,
    maxValue=255,
    adaptiveMethod=cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    thresholdType=cv2.THRESH_BINARY_INV,
    blockSize=11,
    C=2
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title("Grayscale")
axes[0].axis("off")

axes[1].imshow(thresh, cmap="gray")
axes[1].set_title("Adaptive Gaussian Threshold (Inverted)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 3. Contour Detection & Polygon Approximation
We extract external contours, sort by area, and search for the largest closed convex quadrilateral with an aspect ratio consistent with a square.

In [ ]:
contours, _ = cv2.findContours(thresh, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
contours = sorted(contours, key=cv2.contourArea, reverse=True)

image_area = img.shape[0] * img.shape[1]
grid_contour = None
corners = None

for cnt in contours[:100]:
    area = cv2.contourArea(cnt)
    if area < image_area * 0.05:
        continue
    peri = cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
    
    if len(approx) == 4 and cv2.isContourConvex(approx):
        grid_contour = approx
        break

if grid_contour is None:
    raise ValueError("Could not detect Sudoku grid contour. Try a clearer image.")

debug_img = img.copy()
cv2.drawContours(debug_img, [grid_contour], -1, (0, 255, 0), 4)

plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(debug_img, cv2.COLOR_BGR2RGB))
plt.title("Detected Sudoku Grid Boundary")
plt.axis("off")
plt.show()

## 4. Corner Ordering
A perspective homography requires matching source coordinates to standard target destination coordinates in consistent order:
1. Top-Left (minimum sum $x+y$)
2. Top-Right (minimum difference $x-y$)
3. Bottom-Right (maximum sum $x+y$)
4. Bottom-Left (maximum difference $x-y$)

In [ ]:
def order_corners(pts: np.ndarray) -> np.ndarray:
    """Order 4 points as: [Top-Left, Top-Right, Bottom-Right, Bottom-Left]."""
    pts = np.asarray(pts, dtype=np.float32).reshape(4, 2)
    s = pts.sum(axis=1)
    d = np.diff(pts, axis=1).ravel()
    
    return np.array([
        pts[np.argmin(s)],
        pts[np.argmin(d)],
        pts[np.argmax(s)],
        pts[np.argmax(d)]
    ], dtype=np.float32)

corners = order_corners(grid_contour)
print("Ordered Corner Coordinates:")
for label, pt in zip(["Top-Left", "Top-Right", "Bottom-Right", "Bottom-Left"], corners):
    print(f"  {label:12s}: ({pt[0]:.1f}, {pt[1]:.1f})")

## 5. Homography & Perspective Rectification
Using `cv2.getPerspectiveTransform` and `cv2.warpPerspective`, we rectify the angled grid into a top-down $900 \times 900$ image.

In [ ]:
def warp_grid(img: np.ndarray, corners: np.ndarray, size: int = 900):
    """Warp perspective of the detected grid to a canonical size x size square."""
    dst = np.array([
        [0, 0],
        [size - 1, 0],
        [size - 1, size - 1],
        [0, size - 1]
    ], dtype=np.float32)
    
    M = cv2.getPerspectiveTransform(corners.astype(np.float32), dst)
    warped = cv2.warpPerspective(img, M, (size, size))
    return warped, M

warped, M = warp_grid(img, corners, size=900)

plt.figure(figsize=(7, 7))
plt.imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
plt.title("Rectified Top-Down Grid (900×900)")
plt.axis("off")
plt.show()

## 6. Cell Extraction (81 Patches)
We divide the $900 \times 900$ grid into an $9 \times 9$ array of $100 \times 100$ pixel cells.
A safety margin (e.g., 8–10 px) is cropped from all 4 borders of each cell to prevent grid lines from confusing digit classification.

In [ ]:
def split_cells(warped_gray: np.ndarray, margin: int = 8):
    """Split rectified 9x9 grayscale grid into 81 cell image patches."""
    size = warped_gray.shape[0]
    cell_size = size // 9
    cells = []
    
    for row in range(9):
        for col in range(9):
            y1 = row * cell_size + margin
            y2 = (row + 1) * cell_size - margin
            x1 = col * cell_size + margin
            x2 = (col + 1) * cell_size - margin
            cells.append(warped_gray[y1:y2, x1:x2])
            
    return cells

warped_gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
cells = split_cells(warped_gray, margin=8)
print(f"Extracted {len(cells)} cells. Single cell shape: {cells[0].shape}")

# Visualize all 81 extracted cells
fig, axes = plt.subplots(9, 9, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(cells[i], cmap="gray")
    ax.axis("off")
plt.suptitle("81 Extracted Sudoku Cells", fontsize=14, y=0.92)
plt.show()